# Kalshi Pre-Match Brier Score

Measures how well Kalshi's **opening market price** predicts ATP match outcomes.

- **Pre-match prob**: `previous` from the first row (always populated, unlike `open`)
- **Outcome**: last row's `close` (or `previous` if null) → round ≥0.5 to 1, <0.5 to 0
- **One CSV per match**: both files are complementary (p + q ≈ 1), taking both would double-count

In [ ]:
import glob
import os
from collections import defaultdict

import numpy as np
import pandas as pd

DATA_DIR = '../data/2026'

In [2]:
# ── Group files by match ID, pick one per match ───────────────────────────────
# Filename: KXATPMATCH-26FEB01MEDWAW-MED_candles.csv
# Match ID: KXATPMATCH-26FEB01MEDWAW  (everything before last '-{PLAYER}_candles.csv')

all_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_candles.csv')))

by_match = defaultdict(list)
for f in all_files:
    base = os.path.basename(f)                  # KXATPMATCH-26FEB01MEDWAW-MED_candles.csv
    match_id = base.rsplit('-', 1)[0]            # KXATPMATCH-26FEB01MEDWAW
    by_match[match_id].append(f)

# One file per match — first alphabetically
chosen = {mid: sorted(files)[0] for mid, files in by_match.items()}

print(f'Total files   : {len(all_files)}')
print(f'Unique matches: {len(chosen)}')
print(f'Sample match  : {list(chosen.items())[0]}')

Total files   : 1446
Unique matches: 886
Sample match  : ('KXATPCHALLENGERMATCH-26FEB01BOUSCH', 'kalshi_match_data_2026\\KXATPCHALLENGERMATCH-26FEB01BOUSCH-BOU_candles.csv')


In [3]:
# ── Extract pre-match prob and outcome from each file ─────────────────────────

def parse_candle_file(path):
    """
    Returns (first_prob, outcome) or None if file is unusable.
    first_prob : 'previous' from row 0 — the price before trading opened
    outcome    : last non-null close (or previous), rounded to 0/1
    """
    df = pd.read_csv(path)
    if df.empty:
        return None

    # Pre-match probability
    first_prob = df.iloc[0]['previous']
    if pd.isna(first_prob):
        return None

    # Outcome: last non-null close, fallback to last previous
    last_close = df['close'].dropna()
    if not last_close.empty:
        last_val = last_close.iloc[-1]
    else:
        last_val = df['previous'].dropna().iloc[-1]

    outcome = 1 if last_val >= 0.5 else 0
    return float(first_prob), outcome


records = []
skipped = 0

for match_id, path in chosen.items():
    result = parse_candle_file(path)
    if result is None:
        skipped += 1
        continue
    first_prob, outcome = result
    records.append({
        'match_id':   match_id,
        'file':       os.path.basename(path),
        'first_prob': first_prob,
        'outcome':    outcome,
        'brier':      (first_prob - outcome) ** 2,
    })

df = pd.DataFrame(records)
print(f'Parsed: {len(df)}  |  Skipped: {skipped}')
print(df[['file','first_prob','outcome','brier']].head(8).to_string(index=False))

Parsed: 880  |  Skipped: 6
                                              file  first_prob  outcome  brier
KXATPCHALLENGERMATCH-26FEB01BOUSCH-BOU_candles.csv        0.16        0 0.0256
KXATPCHALLENGERMATCH-26FEB01MOCHAZ-HAZ_candles.csv        0.35        1 0.4225
KXATPCHALLENGERMATCH-26FEB01SHITOK-SHI_candles.csv        0.61        1 0.1521
KXATPCHALLENGERMATCH-26FEB02ELLSHI-ELL_candles.csv        0.67        1 0.1089
KXATPCHALLENGERMATCH-26FEB03MORVAS-VAS_candles.csv        0.07        0 0.0049
KXATPCHALLENGERMATCH-26FEB03PRIBON-BON_candles.csv        0.41        0 0.1681
KXATPCHALLENGERMATCH-26FEB03SANCOP-SAN_candles.csv        0.19        1 0.6561
KXATPCHALLENGERMATCH-26FEB03SHAJON-JON_candles.csv        0.44        1 0.3136


In [4]:
# ── Brier score ───────────────────────────────────────────────────────────────

brier = df['brier'].mean()
print(f'Kalshi opening Brier score : {brier:.4f}')
print(f'Random baseline            : 0.2500')
print(f'Skill score                : {1 - brier/0.25:.4f}  (0=random, 1=perfect)')
print()
print(f'Matches analysed : {len(df)}')
print(f'Outcome=1 (won)  : {df["outcome"].sum()}  ({df["outcome"].mean()*100:.1f}%)')
print(f'first_prob mean  : {df["first_prob"].mean():.3f}')
print(f'first_prob range : {df["first_prob"].min():.3f} – {df["first_prob"].max():.3f}')

Kalshi opening Brier score : 0.2293
Random baseline            : 0.2500
Skill score                : 0.0829  (0=random, 1=perfect)

Matches analysed : 880
Outcome=1 (won)  : 456  (51.8%)
first_prob mean  : 0.510
first_prob range : 0.030 – 0.990


In [5]:
# ── Sanity check: MEDWAW match ────────────────────────────────────────────────
# MED file: first_prob ≈ 0.63, outcome = 0 (Medvedev lost to Wawrinka)

medwaw = df[df['match_id'].str.contains('MEDWAW', na=False)]
print('MEDWAW sanity check:')
print(medwaw[['match_id','file','first_prob','outcome']].to_string(index=False))

MEDWAW sanity check:
                match_id                                     file  first_prob  outcome
KXATPMATCH-26FEB01MEDWAW KXATPMATCH-26FEB01MEDWAW-MED_candles.csv        0.63        0


In [6]:
# ── First-prob distribution ───────────────────────────────────────────────────
print('Distribution of opening market probabilities:')
bins = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
counts, _ = np.histogram(df['first_prob'], bins=bins)
for i, c in enumerate(counts):
    lo, hi = bins[i], bins[i+1]
    bar = '|' * c
    print(f'  {lo:.1f}–{hi:.1f}  {c:>4}  {bar}')

Distribution of opening market probabilities:
  0.0–0.1    22  ||||||||||||||||||||||
  0.1–0.2    55  |||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.2–0.3    74  ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.3–0.4   105  |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.4–0.5   140  ||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.5–0.6   145  |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.6–0.7   175  |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||
  0.7–0.8    96  |||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||||

In [7]:
# ── Save results ──────────────────────────────────────────────────────────────
df.to_csv('kalshi_brier_results.csv', index=False)
print('Saved to kalshi_brier_results.csv')

Saved to kalshi_brier_results.csv
